In [1]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

# Keys you still need
for key in ["TAVILY_API_KEY", "LANGSMITH_API_KEY"]:
    print(key, "OK" if os.getenv(key) else "MISSING")

TAVILY_API_KEY OK
LANGSMITH_API_KEY OK


In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",            # use the exact name from `ollama list`
    temperature=0,
    reasoning=False,             # turn off Qwen3's thinking mode
    validate_model_on_init=True, # error right away if the model name is wrong
)

print(llm.invoke("Reply with just the word OK").content)

OK


In [3]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma(
    collection_name="rag-chroma",
    embedding_function=embeddings,
    persist_directory="../chroma_db",
)
retriever = vectorstore.as_retriever()
print("Loaded:", vectorstore._collection.count(), "chunks")   # expect 547

Loaded: 547 chunks


In [4]:
"""from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

urls = [
    # Anthropic engineering: agents, context, tools, MCP
    "https://www.anthropic.com/engineering/building-effective-agents",                  # Dec 2024
    "https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents", # Sep 2025
    "https://www.anthropic.com/engineering/multi-agent-research-system",                # Jun 2025
    "https://www.anthropic.com/engineering/code-execution-with-mcp",                    # Nov 2025
    "https://www.anthropic.com/engineering/writing-tools-for-agents",                   # 2025 (unverified)
    "https://www.anthropic.com/news/contextual-retrieval",                              # Sep 2024, RAG technique
    # Lilian Weng: reasoning, reward hacking, hallucination
    "https://lilianweng.github.io/posts/2025-05-01-thinking/",                          # May 2025
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/",                    # Nov 2024
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",                     # Jul 2024
    # Evals
    "https://hamel.dev/blog/posts/evals-faq/",                                          # expanded Sep 2026
    "https://hamel.dev/blog/posts/evals/",                                              # 2024 classic
    # Agents
    "https://huyenchip.com/2025/01/07/agents.html",                                     # Jan 2025 (unverified)
]

# Load each URL separately so one bad URL doesn't stop the whole cell
docs_list = []
for url in urls:
    try:
        loaded = WebBaseLoader(url).load()
        chars = sum(len(d.page_content) for d in loaded)
        print(f"{chars:>8} chars  {url}")
        if chars < 2000:
            print("          ^ suspiciously short, skipping (blocked page or wrong URL?)")
            continue
        docs_list.extend(loaded)
    except Exception as e:
        print(f"  FAILED  {url} -> {type(e).__name__}: {e}")

print(f"\nLoaded {len(docs_list)} documents")

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(docs_list)
print(f"Split into {len(doc_splits)} chunks")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma(
    collection_name="rag-chroma",
    embedding_function=embeddings,
    persist_directory="../chroma_db",
)
vectorstore.reset_collection()

batch_size = 64
for i in range(0, len(doc_splits), batch_size):
    vectorstore.add_documents(doc_splits[i : i + batch_size])
    print(f"Indexed {min(i + batch_size, len(doc_splits))}/{len(doc_splits)}")

retriever = vectorstore.as_retriever()
print("Total in store:", vectorstore._collection.count())"""

'from langchain_community.document_loaders import WebBaseLoader\nfrom langchain_text_splitters import RecursiveCharacterTextSplitter\nfrom langchain_ollama import OllamaEmbeddings\nfrom langchain_chroma import Chroma\n\nurls = [\n    # Anthropic engineering: agents, context, tools, MCP\n    "https://www.anthropic.com/engineering/building-effective-agents",                  # Dec 2024\n    "https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents", # Sep 2025\n    "https://www.anthropic.com/engineering/multi-agent-research-system",                # Jun 2025\n    "https://www.anthropic.com/engineering/code-execution-with-mcp",                    # Nov 2025\n    "https://www.anthropic.com/engineering/writing-tools-for-agents",                   # 2025 (unverified)\n    "https://www.anthropic.com/news/contextual-retrieval",                              # Sep 2024, RAG technique\n    # Lilian Weng: reasoning, reward hacking, hallucination\n    "https://lilianweng.gi

In [5]:
'''from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")
print(len(embeddings.embed_query("hello world")))   # expect 768'''

'from langchain_ollama import OllamaEmbeddings\nembeddings = OllamaEmbeddings(model="nomic-embed-text")\nprint(len(embeddings.embed_query("hello world")))   # expect 768'

In [6]:
"""lengths = sorted((len(d.page_content), i) for i, d in enumerate(doc_splits))
print("chunks:", len(doc_splits))
print("longest 5 (chars, index):", lengths[-5:])"""

'lengths = sorted((len(d.page_content), i) for i, d in enumerate(doc_splits))\nprint("chunks:", len(doc_splits))\nprint("longest 5 (chars, index):", lengths[-5:])'

In [7]:
import os
from typing import Literal

from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

LOCAL_LLM = os.getenv("LOCAL_LLM", "qwen3:8b")  # add LOCAL_LLM=qwen3:8b to .env


class GradeDocument(BaseModel):
    """Relevance check for a retrieved document."""
    reasoning: str = Field(description="One or two sentences on how the document relates to the question")
    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if the document is relevant to the question, otherwise 'no'"
    )


llm = ChatOllama(model=LOCAL_LLM, temperature=0, reasoning=False)
structured_grader_llm = llm.with_structured_output(GradeDocument, method="json_schema")

grade_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grader assessing whether a retrieved document is relevant to a user question.\n"
     "Grade 'yes' if the document is about the same topic as the question or contains information "
     "that would help answer it, even partially. The document does NOT need to fully answer the question.\n"
     "Grade 'no' only if the document is about a different topic."),
    ("human", "Retrieved document:\n\n{document}\n\nUser question: {question}"),
])

retrieval_grader = grade_prompt | structured_grader_llm

In [8]:
question = "How do AI agents manage memory and context?"
docs = retriever.invoke(question)

for d in docs:
    result = retrieval_grader.invoke({"question": question, "document": d.page_content})
    print(result.binary_score, "|", d.metadata["source"], "|", d.page_content[:80].replace("\n", " "))

# Negative test: an unrelated question should get "no"


yes | https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents | components of context (system prompts, tools, examples, message history, etc) is
yes | https://huyenchip.com/2025/01/07/agents.html | This post covered conceptually how agents work and different components of an ag
yes | https://www.anthropic.com/news/contextual-retrieval | Contextual retrieval in AI systems \ AnthropicSkip to main contentNewsResearchPo
yes | https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents | history, etc).An agent running in a loop generates more and more data that could


In [9]:
off_topic = retrieval_grader.invoke({"question": "What is a good biryani recipe?",
                                     "document": docs[0].page_content})
print("is the question relevant?", off_topic.binary_score)

is the question relevant? no


In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

REFUSAL = (
    "Sorry, I couldn't find information about that in my knowledge base, "
    "so I'm not able to answer it reliably. Feel free to ask about AI agents, "
    "context engineering, LLM reasoning, or evaluation."
)

generate_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions using ONLY the documents provided by the user.\n\n"
     "Rules:\n"
     "1. Every statement in your answer must be supported by the documents. "
     "Do not use outside or prior knowledge, even if you are confident you know the answer.\n"
     "2. The documents come from an automatic search and may be irrelevant to the question. "
     "First decide whether they actually contain the answer.\n"
     f"3. If they do not, reply with exactly: \"{REFUSAL}\" and nothing else.\n"
     "4. If they do, answer in at most three sentences and cite the supporting "
     "document numbers, like [1] or [2][3]."),
    ("human", "<documents>\n{context}\n</documents>\n\nQuestion: {question}"),
])


def format_docs(docs):
    return "\n\n".join(
        f'<document index="{i}" source="{d.metadata.get("source", "unknown")}">\n'
        f"{d.page_content}\n</document>"
        for i, d in enumerate(docs, start=1)
    )


rag_chain = generate_prompt | llm | StrOutputParser()

In [11]:
'''question = "How do AI agents manage memory and context?"
docs = retriever.invoke(question)
generation = rag_chain.invoke({"context": format_docs(docs), "question": question})
print(generation)'''

'question = "How do AI agents manage memory and context?"\ndocs = retriever.invoke(question)\ngeneration = rag_chain.invoke({"context": format_docs(docs), "question": question})\nprint(generation)'

In [12]:
'''question = "Who won the 2022 FIFA World Cup?"
docs = retriever.invoke(question)
print(rag_chain.invoke({"context": format_docs(docs), "question": question}))'''

'question = "Who won the 2022 FIFA World Cup?"\ndocs = retriever.invoke(question)\nprint(rag_chain.invoke({"context": format_docs(docs), "question": question}))'

In [13]:
'''for d in docs:
    print(retrieval_grader.invoke({"question": question, "document": d.page_content}).binary_score)'''

'for d in docs:\n    print(retrieval_grader.invoke({"question": question, "document": d.page_content}).binary_score)'

In [14]:
class GradeHallucination(BaseModel):
    """Whether the answer is supported by the documents."""
    reasoning: str = Field(
        description="Briefly check the answer's main claims against the documents. "
                    "Quote any claim that adds facts not found in the documents, or say all claims are supported."
    )
    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if every factual claim in the answer is supported by the documents, otherwise 'no'"
    )


class GradeAnswer(BaseModel):
    """Whether the answer resolves the question."""
    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if the answer resolves the question, otherwise 'no'"
    )


hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grader checking whether an answer is grounded in a set of documents.\n\n"
     "How to judge:\n"
     "1. Check the meaning of each factual claim in the answer against the documents.\n"
     "2. Paraphrases, summaries, and combinations of information from several documents "
     "count as SUPPORTED, as long as the documents say it.\n"
     "3. Ignore citation numbers like [1] or [2]. Judge only whether the content is in the documents.\n"
     "4. Answer 'no' only if the answer adds a fact, name, number, or detail that is NOT in the documents, "
     "even if that fact is true in the real world."),
    ("human", "<documents>\n{documents}\n</documents>\n\nAnswer to check:\n{generation}"),
])

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grader checking whether an answer resolves a user's question. "
     "Answer 'yes' if it directly addresses and answers the question. "
     "Answer 'no' if it is off-topic, only partially relevant, or declines to answer."),
    ("human", "Question: {question}\n\nAnswer:\n{generation}"),
])

hallucination_grader = hallucination_prompt | llm.with_structured_output(GradeHallucination, method="json_schema")
answer_grader = answer_prompt | llm.with_structured_output(GradeAnswer, method="json_schema")

In [15]:
'''agent_q = "How do AI agents manage memory and context?"
agent_docs = format_docs(retriever.invoke(agent_q))
good_answer = rag_chain.invoke({"context": agent_docs, "question": agent_q})

# 1. Real grounded answer -> hallucination: yes, answer: yes
print("grounded:", hallucination_grader.invoke({"documents": agent_docs, "generation": good_answer}).binary_score)
print("useful:  ", answer_grader.invoke({"question": agent_q, "generation": good_answer}).binary_score)

# 2. True-but-ungrounded answer vs agent docs -> hallucination: no
fake = "Argentina won the 2022 FIFA World Cup, beating France on penalties."
print("argentina grounded:", hallucination_grader.invoke({"documents": agent_docs, "generation": fake}).binary_score)

# 3. Refusal -> answer: no
print("refusal useful:", answer_grader.invoke({"question": agent_q, "generation": REFUSAL}).binary_score)'''

'agent_q = "How do AI agents manage memory and context?"\nagent_docs = format_docs(retriever.invoke(agent_q))\ngood_answer = rag_chain.invoke({"context": agent_docs, "question": agent_q})\n\n# 1. Real grounded answer -> hallucination: yes, answer: yes\nprint("grounded:", hallucination_grader.invoke({"documents": agent_docs, "generation": good_answer}).binary_score)\nprint("useful:  ", answer_grader.invoke({"question": agent_q, "generation": good_answer}).binary_score)\n\n# 2. True-but-ungrounded answer vs agent docs -> hallucination: no\nfake = "Argentina won the 2022 FIFA World Cup, beating France on penalties."\nprint("argentina grounded:", hallucination_grader.invoke({"documents": agent_docs, "generation": fake}).binary_score)\n\n# 3. Refusal -> answer: no\nprint("refusal useful:", answer_grader.invoke({"question": agent_q, "generation": REFUSAL}).binary_score)'

In [16]:
# Hallucination grader check: Q1's first attempt must pass, made-up content must still fail
'''q = "How do AI agents manage memory and context?"

docs = retriever.invoke(q)
relevant = [d for d in docs
            if retrieval_grader.invoke({"question": q, "document": d.page_content}).binary_score == "yes"]
ctx = format_docs(relevant)

ans = rag_chain.invoke({"context": ctx, "question": q})
print("ANSWER:\n", ans)

h = hallucination_grader.invoke({"documents": ctx, "generation": ans})
print("\nQ1 grounded:", h.binary_score, "| expected: yes")
print("reason:", h.reasoning)

fake = "Argentina won the 2022 FIFA World Cup, beating France on penalties."
h2 = hallucination_grader.invoke({"documents": ctx, "generation": fake})
print("\nArgentina grounded:", h2.binary_score, "| expected: no")
print("reason:", h2.reasoning)
'''

'q = "How do AI agents manage memory and context?"\n\ndocs = retriever.invoke(q)\nrelevant = [d for d in docs\n            if retrieval_grader.invoke({"question": q, "document": d.page_content}).binary_score == "yes"]\nctx = format_docs(relevant)\n\nans = rag_chain.invoke({"context": ctx, "question": q})\nprint("ANSWER:\n", ans)\n\nh = hallucination_grader.invoke({"documents": ctx, "generation": ans})\nprint("\nQ1 grounded:", h.binary_score, "| expected: yes")\nprint("reason:", h.reasoning)\n\nfake = "Argentina won the 2022 FIFA World Cup, beating France on penalties."\nh2 = hallucination_grader.invoke({"documents": ctx, "generation": fake})\nprint("\nArgentina grounded:", h2.binary_score, "| expected: no")\nprint("reason:", h2.reasoning)\n'

In [17]:
class RouteQuery(BaseModel):
    """Where to send the user's question."""
    datasource: Literal["vectorstore", "web_search"] = Field(
        description="'vectorstore' for questions about the knowledge base topics, otherwise 'web_search'"
    )


router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You route a user question to either a vectorstore or web search.\n\n"
     "The vectorstore contains articles on: AI agents and agent design patterns, "
     "context engineering, multi-agent systems, tool design for agents and MCP, "
     "RAG and contextual retrieval, LLM reasoning and test-time compute, "
     "reward hacking, hallucinations in LLMs, and LLM evaluation (evals).\n\n"
     "Rules:\n"
     "1. Use 'vectorstore' for questions related to these topics. "
     "Be lenient: the question does not need to use the exact keywords.\n"
     "2. Use 'web_search' for anything else, and for questions about recent news, "
     "current events, or the latest releases, even if the topic matches, "
     "because the vectorstore is a fixed snapshot."),
    ("human", "{question}"),
])

question_router = router_prompt | llm.with_structured_output(RouteQuery, method="json_schema")

In [18]:
'''tests = {
    "How do AI agents manage memory and context?": "vectorstore",
    "What is LLM-as-a-judge?":                      "vectorstore",
    "Why do reasoning models think before answering?": "vectorstore",
    "Who won the 2022 FIFA World Cup?":             "web_search",
    "What's the weather in Karachi today?":         "web_search",
    "What AI models were released this week?":      "web_search",   # topic matches, but recency
}

for q, expected in tests.items():
    got = question_router.invoke({"question": q}).datasource
    print("✅" if got == expected else "❌", f"{got:<12} | {q}")'''

'tests = {\n    "How do AI agents manage memory and context?": "vectorstore",\n    "What is LLM-as-a-judge?":                      "vectorstore",\n    "Why do reasoning models think before answering?": "vectorstore",\n    "Who won the 2022 FIFA World Cup?":             "web_search",\n    "What\'s the weather in Karachi today?":         "web_search",\n    "What AI models were released this week?":      "web_search",   # topic matches, but recency\n}\n\nfor q, expected in tests.items():\n    got = question_router.invoke({"question": q}).datasource\n    print("✅" if got == expected else "❌", f"{got:<12} | {q}")'

In [19]:
from langchain_core.documents import Document
from langchain_tavily import TavilySearch

web_search_tool = TavilySearch(max_results=3, topic="general")


def web_search(query: str) -> list[Document]:
    """Search the web and return results as Documents, same shape as the retriever."""
    response = web_search_tool.invoke({"query": query})
    return [
        Document(
            page_content=r["content"],
            metadata={"source": r["url"], "title": r.get("title", "")},
        )
        for r in response["results"]
    ]

In [20]:
# ---------- Config + query rewriter ----------
ALLOW_WEB_FALLBACK = os.getenv("ALLOW_WEB_FALLBACK", "true").lower() == "true"
MAX_REWRITES = 1

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You rewrite a user question into a better search query for a vector database of "
     "articles about AI agents, context engineering, multi-agent systems, tools and MCP, "
     "RAG, LLM reasoning, reward hacking, hallucinations, and LLM evaluation.\n"
     "Keep the original meaning. Use specific technical terms the articles would use. "
     "Do not answer the question. Output only the rewritten query, nothing else."),
    ("human", "{question}"),
])

query_rewriter = rewrite_prompt | llm | StrOutputParser()

# quick test: should print a search query, not an answer
print(query_rewriter.invoke({"question": "how do bots remember stuff between chats?"}))


How do AI agents maintain and retain contextual information across multiple conversational interactions using memory mechanisms in multi-agent systems and context engineering techniques?


In [21]:
question = "Who won the 2022 FIFA World Cup?"
web_docs = web_search(question)

for d in web_docs:
    print(d.metadata["source"], "|", d.page_content[:100].replace("\n", " "))

print("\n->", rag_chain.invoke({"context": format_docs(web_docs), "question": question}))

https://en.wikipedia.org/wiki/2022_FIFA_World_Cup | The final was played on 18 December between Argentina and France. Both teams had won the event twice
https://en.wikipedia.org/wiki/2022_FIFA_World_Cup_final | | Gonzalo Montiel's match-winning penalty kick in the penalty shoot-out | | Event | 2022 FIFA World 
https://www.cbs8.com/article/sports/soccer/world-cup/who-won-last-world-cup/507-38e0e34c-dd9c-435f-ad9c-227eaa9f6d7a | This story continues below.  More in Soccer       England fans celebrate World Cup victory over Norw

-> Argentina won the 2022 FIFA World Cup, defeating France in a penalty shootout after the scores were tied 3-3 [3]. This victory marked Argentina's third World Cup title [2].


In [22]:
from typing import List
from typing_extensions import TypedDict
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END


class GraphState(TypedDict):
    question: str            # the user's original question (never changes)
    search_query: str        # NEW: what we actually search with (may be rewritten)
    generation: str
    documents: List[Document]
    web_searched: bool
    rewrites: int            # NEW: how many rewrites so far
    answer_source: str       # NEW: "knowledge_base", "web", or "none"


# ---------- Fallback policy: one place decides what to try next ----------

def next_fallback(state):                                        # NEW
    if state.get("rewrites", 0) < MAX_REWRITES and not state.get("web_searched"):
        return "rewrite"
    if ALLOW_WEB_FALLBACK and not state.get("web_searched"):
        return "web_search"
    return "give_up"


# ---------- Nodes ----------

def retrieve(state):
    query = state.get("search_query") or state["question"]        # NEW
    print(f"---RETRIEVE--- ({query})")
    return {"documents": retriever.invoke(query)}


def grade_documents(state):
    print("---GRADE DOCUMENTS---")
    relevant = []
    for d in state["documents"]:
        grade = retrieval_grader.invoke(
            {"question": state["question"], "document": d.page_content}
        ).binary_score
        print(f"  {grade}: {d.metadata.get('source')}")
        if grade == "yes":
            relevant.append(d)
    return {"documents": relevant}


def rewrite(state):                                              # NEW
    new_query = query_rewriter.invoke({"question": state["question"]}).strip()
    print(f"---REWRITE QUERY--- -> {new_query}")
    return {"search_query": new_query, "rewrites": state.get("rewrites", 0) + 1}


def web_search_node(state):
    print("---WEB SEARCH---")
    web_docs = web_search(state["question"])
    return {"documents": state.get("documents", []) + web_docs, "web_searched": True}


def generate(state):
    print("---GENERATE---")
    generation = rag_chain.invoke(
        {"context": format_docs(state["documents"]), "question": state["question"]}
    )
    source = "web" if state.get("web_searched") else "knowledge_base"   # NEW
    return {"generation": generation, "answer_source": source}


def give_up(state):
    print("---GIVE UP: RETURN REFUSAL---")
    return {"generation": REFUSAL, "answer_source": "none"}


# ---------- Edges ----------

def route_question(state):
    print("---ROUTE QUESTION---")
    source = question_router.invoke({"question": state["question"]}).datasource
    if source == "web_search" and not ALLOW_WEB_FALLBACK:         # NEW: scoped mode
        print("  -> out of scope (web disabled)")
        return "give_up"
    print("  ->", source)
    return source


def decide_to_generate(state):
    if not state["documents"]:
        nxt = next_fallback(state)                                # NEW
        print("---NO RELEVANT DOCUMENTS ->", nxt.upper(), "---")
        return nxt
    print("---RELEVANT DOCUMENTS FOUND -> GENERATE---")
    return "generate"


def check_generation(state):
    print("---CHECK GENERATION---")
    question, generation = state["question"], state["generation"]
    fallback = next_fallback(state)                               # NEW

    if generation.strip() == REFUSAL:
        print("  refusal ->", fallback)
        return fallback

    h = hallucination_grader.invoke(
        {"documents": format_docs(state["documents"]), "generation": generation}
    )
    if h.binary_score == "no":
        print("  not grounded ->", fallback, "|", h.reasoning)
        return fallback

    useful = answer_grader.invoke({"question": question, "generation": generation}).binary_score
    if useful == "no":
        print("  not useful ->", fallback)
        return fallback

    print("  grounded and useful -> done")
    return "done"


# ---------- Build the graph ----------

workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("rewrite", rewrite)                             # NEW
workflow.add_node("web_search", web_search_node)
workflow.add_node("generate", generate)
workflow.add_node("give_up", give_up)

workflow.add_conditional_edges(START, route_question,
    {"vectorstore": "retrieve", "web_search": "web_search", "give_up": "give_up"})
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges("grade_documents", decide_to_generate,
    {"generate": "generate", "rewrite": "rewrite", "web_search": "web_search", "give_up": "give_up"})
workflow.add_edge("rewrite", "retrieve")                          # NEW: rewrite loops back to retrieval
workflow.add_edge("web_search", "generate")
workflow.add_conditional_edges("generate", check_generation,
    {"done": END, "rewrite": "rewrite", "web_search": "web_search", "give_up": "give_up"})
workflow.add_edge("give_up", END)

app = workflow.compile()


In [23]:
# ---------- Test both modes: research (web on) vs scoped (web off) ----------
def run(q):
    out = app.invoke({"question": q, "web_searched": False, "rewrites": 0})
    print(f"\n>>> SOURCE: {out['answer_source']}\n>>> ANSWER: {out['generation']}\n")

tests = [
    "How do AI agents manage memory and context?",                     # KB, no rewrite
    "how do bots remember stuff between chats?",                       # vague wording -> may rewrite, should end in KB
    "Who won the 2022 FIFA World Cup?",                                # research: web | scoped: refuse
    "What is the Kowalski-Reyes benchmark for agent tool selection?",  # must refuse in both
]

for mode in [True, False]:
    ALLOW_WEB_FALLBACK = mode
    print("\n" + "#" * 80 + f"\n# MODE: {'research (web on)' if mode else 'scoped (web off)'}\n" + "#" * 80)
    for q in tests:
        print("\n" + "=" * 70 + f"\nQ: {q}")
        run(q)



################################################################################
# MODE: research (web on)
################################################################################

Q: How do AI agents manage memory and context?
---ROUTE QUESTION---
  -> vectorstore
---RETRIEVE--- (How do AI agents manage memory and context?)
---GRADE DOCUMENTS---
  yes: https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents
  yes: https://huyenchip.com/2025/01/07/agents.html
  yes: https://www.anthropic.com/news/contextual-retrieval
  yes: https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents
---RELEVANT DOCUMENTS FOUND -> GENERATE---
---GENERATE---
---CHECK GENERATION---
  grounded and useful -> done

>>> SOURCE: knowledge_base
>>> ANSWER: AI agents manage memory and context by dynamically retrieving information at runtime and maintaining a "just in time" approach to context, which involves supplementing the model's context with a memory 

In [24]:
tests = [
    "How do AI agents manage memory and context?",   # vectorstore -> generate -> done
    "Who won the 2022 FIFA World Cup?",              # router -> web search -> done
    "What is LLM-as-a-judge?",                       # vectorstore, eval docs
]

for q in tests:
    print("\n" + "=" * 80 + f"\nQ: {q}")
    result = app.invoke({"question": q, "web_searched": False})
    print("\nANSWER:", result["generation"])


Q: How do AI agents manage memory and context?
---ROUTE QUESTION---
  -> vectorstore
---RETRIEVE--- (How do AI agents manage memory and context?)
---GRADE DOCUMENTS---
  yes: https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents
  yes: https://huyenchip.com/2025/01/07/agents.html
  yes: https://www.anthropic.com/news/contextual-retrieval
  yes: https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents
---RELEVANT DOCUMENTS FOUND -> GENERATE---
---GENERATE---
---CHECK GENERATION---
  grounded and useful -> done

ANSWER: AI agents manage memory and context by dynamically retrieving information at runtime and maintaining a "just in time" approach to context, which involves supplementing the model's context with a memory system to handle information that exceeds the model's context limit [2]. Context engineering is also crucial, as it involves curating the limited context window from an evolving universe of information, treating context 

In [25]:
# ── Mechanical grounding check ───────────────────────────────────────────────
# A second, independent opinion on grounding that does not involve an LLM.
#
# Why bother, when hallucination_grader already exists? Because that grader is
# itself a model, it is asked to be generous about paraphrase, and when it is
# wrong it is wrong in the direction that hurts: it says "yes".
#
# This check is lexical, so it CANNOT read meaning. A correct paraphrase will
# score low here. That is the trade: it is a tripwire, not a verdict. A sentence
# it flags is one to look at by hand.
#
# The key design choice: overlap is measured against ONE document at a time,
# never the pooled corpus. A claim assembled from words scattered across four
# passages is exactly the failure mode worth catching, and pooling would hide it.

import re

_STOP = set("""
a an the and or but if then than that this these those of in on at to for from by with
without about into over under again further is are was were be been being am do does did
doing have has had having i you he she it we they them his her its our your their what
which who whom when where why how all any both each few more most other some such no nor
not only own same so too very can will just should now as it's also may might must would
could one two use used using
""".split())

def _words(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower())
            if w not in _STOP and len(w) > 2]

def _sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if len(_words(p)) >= 4]

def grounding_report(generation, docs, threshold=0.55, verbose=True):
    """Per sentence: the best single-document content-word overlap.

    Returns (worst_score, rows). `rows` is (sentence, best_score, best_doc_index).
    """
    if generation.strip() == REFUSAL:
        if verbose:
            print("  (refusal — nothing to check)")
        return 1.0, []

    doc_words = [set(_words(d.page_content)) for d in docs]
    rows = []

    for sent in _sentences(generation):
        sw = set(_words(sent))
        if not sw:
            continue
        best, best_i = 0.0, -1
        for i, dw in enumerate(doc_words):
            score = len(sw & dw) / len(sw)
            if score > best:
                best, best_i = score, i
        rows.append((sent, best, best_i))

    if verbose:
        for sent, score, i in rows:
            mark = "ok  " if score >= threshold else "WEAK"
            where = f"doc[{i + 1}]" if i >= 0 else "nothing"
            print(f"  {mark} {score:.2f} vs {where}: {sent[:88]}")

    worst = min((r[1] for r in rows), default=1.0)
    return worst, rows


In [26]:
# ── Probe one question down BOTH paths, honestly ─────────────────────────────
# The earlier version of this cell printed the vectorstore chunks, then answered
# from web search, and labelled neither. Comparing that answer against those
# chunks made a correctly-grounded answer look invented.
#
# The rule this cell now follows: print the documents the answer was ACTUALLY
# generated from, right next to the answer. Never any other set.

def probe(question, docs, label):
    print(f"\n{'=' * 70}\n{label}\n{'=' * 70}")

    if not docs:
        print("no documents — nothing to generate from")
        return

    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "unknown")
        print(f"\n[{i}] {src}\n    {d.page_content[:300].replace(chr(10), ' ')}")

    ctx = format_docs(docs)
    ans = rag_chain.invoke({"context": ctx, "question": question})

    print(f"\nANSWER:\n{ans}\n")
    h = hallucination_grader.invoke({"documents": ctx, "generation": ans})
    print(f"LLM GROUNDED : {h.binary_score}  —  {h.reasoning[:200]}")
    print("MECHANICAL   :")
    worst, _ = grounding_report(ans, docs)
    print(f"  worst sentence overlap: {worst:.2f}")
    if h.binary_score == "yes" and worst < 0.55:
        print("  ^^ grader said yes but a sentence has thin lexical support — read it.")


q = "What is LLM-as-a-judge?"
print("ROUTER ->", question_router.invoke({"question": q}).datasource)

probe(q, retriever.invoke(q), "VECTORSTORE PATH  (answered from these chunks)")
probe(q, web_search(q), "WEB PATH  (answered from these results)")


ROUTER -> vectorstore

VECTORSTORE PATH  (answered from these chunks)

[1] https://hamel.dev/blog/posts/evals-faq/
    Q: What should I do when I can’t get my LLM judge to agree with human reviewers? To debug a LLM judge, you need examples with human Pass/Fail labels to compare its decisions against. An effective way to get these labels is error analysis, which provides you with a structured way to review your appli

[2] https://hamel.dev/blog/posts/evals-faq/
    Q: How much context should I give a LLM judge? Give each judge only the parts of the trace it needs for its failure mode. Do not give every judge the same full trace by default. Extra context can cause context rot and make the judge worse. Finding the right pieces of context often requires experimen

[3] https://hamel.dev/blog/posts/evals-faq/
    When using an LLM, starting with a powerful model can make it easier to develop the judge’s prompt. Once it works well, try smaller, cheaper models and measure how much accuracy you

In [27]:
# ── Does the graph refuse when it should? ────────────────────────────────────
# The only honest test of a refusal path is a question whose answer is in the
# model's head but not in the corpus. If it answers with citations anyway, every
# gap will be filled that way and none of the citations can be trusted.
#
# These run through `app`, not by hand, so what is tested is the shipped wiring:
# route -> retrieve -> grade -> generate -> check -> (web search | give up).

probes = [
    # Squarely in the corpus — should answer from the vectorstore.
    "How do AI agents manage memory and context?",
    # Famous fact, certainly not in these articles. The web branch may rescue it;
    # what matters is that it is never answered from the vectorstore chunks.
    "Who won the 2022 FIFA World Cup?",
    # Plausible-sounding but invented, so the web cannot rescue it either.
    # This one must end in the refusal.
    "What is the Kowalski-Reyes benchmark for agent tool selection?",
]

for q in probes:
    print("\n" + "#" * 70)
    print("#", q)
    print("#" * 70)
    out = app.invoke({"question": q, "web_searched": False})

    ans = out["generation"]
    refused = ans.strip() == REFUSAL
    print(f"\nREFUSED: {refused}")
    print(f"ANSWER : {ans}\n")

    if not refused:
        print("grounding of that answer against the docs it used:")
        worst, _ = grounding_report(ans, out["documents"])
        print(f"  worst sentence overlap: {worst:.2f}")



######################################################################
# How do AI agents manage memory and context?
######################################################################
---ROUTE QUESTION---
  -> vectorstore
---RETRIEVE--- (How do AI agents manage memory and context?)
---GRADE DOCUMENTS---
  yes: https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents
  yes: https://huyenchip.com/2025/01/07/agents.html
  yes: https://www.anthropic.com/news/contextual-retrieval
  yes: https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents
---RELEVANT DOCUMENTS FOUND -> GENERATE---
---GENERATE---
---CHECK GENERATION---
  grounded and useful -> done

REFUSED: False
ANSWER : AI agents manage memory and context by dynamically retrieving information at runtime and maintaining a "just in time" approach to context, which involves supplementing the model's context with a memory system to handle information that exceeds the model's context l